<a href="https://colab.research.google.com/github/FC-Andrade/Analises-complementares/blob/main/AN%C3%81LISE_DOS_MODOS_NORMAIS_(NMA)_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ================================================================
# Análise de Modos Normais (NMA) automatizada para triplicatas
# ================================================================

# Instalar dependências
!pip install -q prody biopython matplotlib numpy seaborn py3Dmol tqdm MDAnalysis

import prody as pr
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from google.colab import files
import MDAnalysis as mda
from tqdm import tqdm
import os

WORKDIR = "/content/nma_triplicates"
os.makedirs(WORKDIR, exist_ok=True)
os.chdir(WORKDIR)

In [ ]:
# 1. Upload das topologias PDB e trajetórias XTC
print("📂 Faça upload dos arquivos PDB (topologias) e XTC (trajetórias) das triplicatas (Apo e AG73).")
uploaded = files.upload()
uploaded_files = list(uploaded.keys())

In [ ]:
# Identificar PDB e XTC
pdb_files = [f for f in uploaded_files if f.endswith('.pdb')]
xtc_files = [f for f in uploaded_files if f.endswith(('.xtc', '.trr', '.dcd'))]

if len(pdb_files) != len(xtc_files):
    print("⚠️ Atenção: número de topologias e trajetórias não coincide!")

# 2. Separar sistemas por tipo (Apo ou AG73)
systems = {"Apo": [], "AG73": []}
for pdb, xtc in zip(pdb_files, xtc_files):
    if "Apo" in pdb:
        systems["Apo"].append((pdb, xtc))
    else:
        systems["AG73"].append((pdb, xtc))
print(f"Sistemas detectados: {{k: len(v) for k,v in systems.items()}}")

# 3. Função para extrair frames inicial e final e salvar PDB
def extract_frames(top_file, traj_file, frame_initial=0, frame_final=-1):
    u = mda.Universe(top_file, traj_file)
    # Extrair frame inicial
    u.trajectory[frame_initial]
    pdb_init = f"{top_file.replace('.pdb','')}_init.pdb"
    with mda.Writer(pdb_init) as W:
        W.write(u)
    # Extrair frame final
    u.trajectory[frame_final]
    pdb_final = f"{top_file.replace('.pdb','')}_final.pdb"
    with mda.Writer(pdb_final) as W:
        W.write(u)
    return pdb_init, pdb_final



In [ ]:
# 4. Processar todas as réplicas
results_dir = "NMA_results"
os.makedirs(results_dir, exist_ok=True)

for sys_name, files_list in systems.items():
    for i, (pdb, xtc) in enumerate(files_list, start=1):
        print(f"\n🔹 Processando {sys_name} réplica {i}: {pdb} / {xtc}")
        pdb_init, pdb_final = extract_frames(pdb, xtc)

        # Carregar e alinhar
        ref = pr.parsePDB(pdb_init)
        mob = pr.parsePDB(pdb_final)
        ref_ca = ref.select('protein and name CA')
        mob_ca = mob.select('protein and name CA')
        pr.calcTransformation(mob_ca, ref_ca).apply(mob_ca)

        # NMA
        anm = pr.ANM(f"{sys_name}_rep{i}")
        anm.buildHessian(ref_ca)
        anm.calcModes(n_modes=12)

        # Vetor de deformação
        deform_vector = pr.calcDeformVector(ref_ca, mob_ca)
        overlaps = [abs(pr.calcOverlap(mode, deform_vector)) for mode in anm]
        cum_overlap = np.cumsum(overlaps)/np.sum(overlaps)

        # Gráfico overlap
        sns.set(style="whitegrid", font_scale=1.2)
        plt.figure(figsize=(7,5))
        plt.bar(range(1,len(overlaps)+1), overlaps, color='#1f77b4', label=sys_name)
        plt.plot(range(1,len(overlaps)+1), cum_overlap, '-o', color='#ff7f0e', label='Cumulativo')
        plt.xlabel("Mode index")
        plt.ylabel("Overlap")
        plt.title(f"Overlap NMA - {sys_name} réplica {i}")
        plt.legend()
        plt.tight_layout()
        fig_name = os.path.join(results_dir, f"{sys_name}_rep{i}_overlap.png")
        plt.savefig(fig_name, dpi=300)
        plt.show()

        # Morphing primeiros 4 modos
        ensemble = pr.generatePerturbedCoords(ref_ca, anm[:4], amplitude=2.0)
        morph_pdb = os.path.join(results_dir, f"{sys_name}_rep{i}_morph.pdb")
        pr.writePDB(morph_pdb, ensemble)
        print(f"✅ Morph salvo: {morph_pdb}")

print("\n✅ NMA completa para todas as triplicatas. Resultados em:", results_dir)
